%% [markdown]
# RLHF Data Generation with Trained LoRA
This notebook systematically generates image sets for Reinforcement Learning from Human Feedback (RLHF).

**Key steps:**
1.  Load the base Stable Diffusion model and our trained LoRA adapter.
2.  Define a list of prompts for which to generate images.
3.  For each prompt, generate multiple image samples.
4.  For each sample, save the final image, the initial latent noise, and all generation metadata.
---

In [1]:
# %%
# Essential Imports
import gc
import json
import hashlib
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional
from PIL import Image as PILImage
import torch
from diffusers import StableDiffusionPipeline
from tqdm.auto import tqdm

In [ ]:
# %%
# --- Setup and Path Configuration ---

# The directory where you want to save the generated data for RLHF
RLHF_DATA_DIR = Path("./rlhf_generation_data")
RLHF_DATA_DIR.mkdir(exist_ok=True)

# Path to your trained LoRA model
LORA_CHECKPOINT_BASE_DIR = Path("./lora_training_runs")
RESUME_RUN_ID = '519037_run_20250903-160027' # <-- Make sure this is your latest run ID
LORA_PATH = LORA_CHECKPOINT_BASE_DIR / RESUME_RUN_ID / "lora_checkpoints/best_lora_adapter"

In [ ]:
# %%
# --------------------------------------------------------------------------------------------------
## Core Generation Functions (Refactored for RLHF)
# --------------------------------------------------------------------------------------------------

def load_lora_for_inference(
    lora_adapter_path: Path,
    device: str = 'cuda'
) -> Optional[StableDiffusionPipeline]:
    """Loads the Stable Diffusion pipeline and applies the trained LoRA weights."""
    if not lora_adapter_path.is_dir():
        print(f"❌ Error: LoRA adapter path must be a directory. Path not found at {lora_adapter_path}.")
        return None

    print("🚀 Loading base Stable Diffusion model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False
    )

    print(f"✅ Applying LoRA weights from {lora_adapter_path}")
    pipe.unet.load_attn_procs(lora_adapter_path)
    pipe = pipe.to(device)
    return pipe


def generate_and_save_samples(
    pipe: StableDiffusionPipeline,
    prompt: str,
    output_dir: Path,
    num_samples: int = 3,
    num_inference_steps: int = 150,
    guidance_scale: float = 7.5,
):
    """
    Generates multiple samples for a single prompt and saves the image, latent, and metadata.
    """
    pipe.unet.eval()
    
    # Get dimensions from the model config
    height = pipe.unet.config.sample_size * pipe.vae_scale_factor
    width = pipe.unet.config.sample_size * pipe.vae_scale_factor
    device = pipe.device
    
    for i in range(num_samples):
        # Generate a unique seed for each sample
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
        generator = torch.Generator(device=device).manual_seed(seed)

        # 💡 CRITICAL: Create the initial latent noise BEFORE the pipe call
        latents = torch.randn(
            (1, pipe.unet.config.in_channels, int(height // 8), int(width // 8)),
            generator=generator,
            device=device,
            dtype=torch.float16
        )

        # Generate the image using the pre-defined latent
        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type):
                result = pipe(
                    prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    generator=generator,
                    latents=latents, # <-- Pass the exact latent tensor
                )
        image = result.images[0]

        # --- Save all necessary artifacts ---
        sample_base_name = f"sample_{i}"
        
        # 1. Save the final image
        image.save(output_dir / f"{sample_base_name}.png")

        # 2. Save the initial latent tensor (most important for DPO)
        # We move it to cpu to save space and avoid GPU memory issues
        torch.save(latents.cpu(), output_dir / f"{sample_base_name}_latent.pt")

        # 3. Save all metadata
        metadata = {
            "prompt": prompt,
            "seed": seed,
            "num_inference_steps": num_inference_steps,
            "guidance_scale": guidance_scale,
            "image_path": str(output_dir / f"{sample_base_name}.png"),
            "latent_path": str(output_dir / f"{sample_base_name}_latent.pt"),
        }
        with open(output_dir / f"{sample_base_name}_meta.json", "w") as f:
            json.dump(metadata, f, indent=4)

def generate_rlhf_dataset(
    pipe: StableDiffusionPipeline,
    prompts: List[str],
    base_output_dir: Path,
    num_samples_per_prompt: int = 3,
    ):
    """Orchestrates the generation of the entire RLHF dataset."""
    print(f"✨ Starting RLHF data generation for {len(prompts)} prompts...")
    
    for prompt in tqdm(prompts, desc="Generating prompts"):
        # Create a unique, clean directory name for each prompt using a hash
        prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()[:10]
        prompt_dir = base_output_dir / prompt_hash
        prompt_dir.mkdir(exist_ok=True)
        
        generate_and_save_samples(
            pipe=pipe,
            prompt=prompt,
            output_dir=prompt_dir,
            num_samples=num_samples_per_prompt,
        )
        
    print(f"\n✅ Generation complete! Data saved in: {base_output_dir}")


def unload_model(pipe: StableDiffusionPipeline) -> None:
    """Unloads the pipeline and clears GPU memory."""
    print("🔻 Unloading pipeline to free GPU memory...")
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ Model unloaded and GPU cache cleared.")

In [ ]:
# %%
# --------------------------------------------------------------------------------------------------
## Main Execution Block
# --------------------------------------------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load LoRA Weights and Model ----
pipeline = load_lora_for_inference(LORA_PATH, device)

if pipeline is None:
    print("❌ Failed to load the model. Halting execution.")
else:
    # ---- Define the Prompts for Generation ----
    # Instead of a random sample, define a structured list of prompts
    # This gives you control over the data you collect.
    prompts_to_generate = [
        "A 25 years old asian female smiling happily",
        "A 25 years old asian female with a neutral expression",
        "A 60 years old caucasian male showing a subtle emotion",
        "A 60 years old caucasian male smiling happily",
        "A 5 years old afroamerican female with a neutral expression",
        "A 45 years old asian male smiling slightly",
        "A 80 years old caucasian female with a neutral expression",
    ]

    # ---- Run the Dataset Generation ----
    generate_rlhf_dataset(
        pipe=pipeline,
        prompts=prompts_to_generate,
        base_output_dir=RLHF_DATA_DIR,
        num_samples_per_prompt=3 # Generate 3 images for A/B/C comparison
    )

    # ---- Unload Model ----
    unload_model(pipeline)